# Pic2Word (CVPR 2023) — bản mini chạy Kaggle GPU

Bài gốc: **Saito et al., "Pic2Word: Mapping Pictures to Words for Zero-shot Composed Image Retrieval", CVPR 2023** ([code gốc](https://github.com/google-research/composed_image_retrieval)).

Ý tưởng: train một **mapping network** biến CLIP image embedding → "pseudo-word" token trong không gian text, sao cho câu chứa pseudo-word có embedding khớp chính ảnh đó. Từ đó truy vấn *ảnh + chữ* (ZS-CIR) không cần dữ liệu triplet.

Bản mini này (phục vụ học phần Xử lý ảnh):
- CLIP ViT-B/32 **đóng băng**, mapper = MLP 2 lớp (512→512).
- Train trên **2000 ảnh Flickr8k** (1 caption/ảnh, pseudo-label smoothing 0.2 như bài gốc), 2 epoch ≈ 10 phút GPU.
- Eval **self-retrieval** trên 200 ảnh held-out (mapper vs nhét raw image embedding), demo truy xuất.
- Kernel chạy headless: Hermes push bằng `kaggle kernels push`, Kaggle tự chạy hết các cell rồi tắt VM.

In [ ]:
import random, numpy as np, torch

CFG = dict(
    MODEL="openai/clip-vit-base-patch32",
    MAX_TRAIN_IMAGES=2000,
    N_GALLERY=200,
    EPOCHS=2,
    BATCH=64,
    LR=1e-4,
    SMOOTH_P=0.2,      # pseudo-label smoothing: thay caption bằng caption của ảnh khác
    SEED=42,
    TEMPLATE="A photo of .",   # dấu "." (vị trí 1 sau BOS) bị thay bằng pseudo-token
)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
!pip install -q "transformers>=4.35"
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer

clip = CLIPModel.from_pretrained(CFG["MODEL"]).to(DEVICE).eval()
for p in clip.parameters():
    p.requires_grad_(False)
tok = AutoTokenizer.from_pretrained(CFG["MODEL"])
proc = CLIPProcessor.from_pretrained(CFG["MODEL"])
D = clip.config.projection_dim
print("CLIP ok | projection_dim =", D)

## 1. Dữ liệu Flickr8k (dataset gắn kèm kernel)

In [ ]:
import glob, os
import pandas as pd

caps = glob.glob("/kaggle/input/**/captions.txt", recursive=True)
assert caps, "KHÔNG thấy captions.txt — kiểm tra dataset_sources trong kernel-metadata.json"
CAP = caps[0]
IMG_DIR = os.path.join(os.path.dirname(CAP), "Images")
print("captions:", CAP, "| images dir:", IMG_DIR, "tồn tại:", os.path.isdir(IMG_DIR))

rows = pd.read_csv(CAP)
if not {"image", "caption"} <= set(rows.columns):  # fallback parse thủ công
    lines = open(CAP, encoding="utf-8").read().splitlines()[1:]
    recs = []
    for l in lines:
        if "," in l:
            img, cap = l.split(",", 1)
            recs.append((img.strip(), cap.strip()))
    rows = pd.DataFrame(recs, columns=["image", "caption"])

img_cap = rows.dropna().groupby("image").first().reset_index()  # 1 caption/ảnh
img_cap = img_cap.sample(frac=1.0, random_state=CFG["SEED"]).reset_index(drop=True)
img_cap["path"] = img_cap["image"].apply(lambda s: os.path.join(IMG_DIR, s))
img_cap = img_cap[img_cap["path"].apply(os.path.exists)].reset_index(drop=True)
print("Ảnh có sẵn:", len(img_cap), "| caption mẫu:", img_cap["caption"].iloc[0][:80])

train_df = img_cap.iloc[:CFG["MAX_TRAIN_IMAGES"]].reset_index(drop=True)
gallery_df = img_cap.iloc[CFG["MAX_TRAIN_IMAGES"]:CFG["MAX_TRAIN_IMAGES"] + CFG["N_GALLERY"]].reset_index(drop=True)
print("train:", len(train_df), "| gallery (held-out):", len(gallery_df))

In [ ]:
import torch.nn as nn
from PIL import Image

# Mapping network: image embedding (joint space) -> pseudo-word token embedding
torch.manual_seed(CFG["SEED"])
mapper = nn.Sequential(
    nn.Linear(D, D), nn.LayerNorm(D), nn.GELU(), nn.Linear(D, D),
).to(DEVICE)
print("mapper params:", sum(p.numel() for p in mapper.parameters()))

@torch.no_grad()
def encode_images(paths, bs=64):
    feats = []
    for i in range(0, len(paths), bs):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i + bs]]
        inputs = proc(images=imgs, return_tensors="pt").to(DEVICE)
        feats.append(clip.get_image_features(**inputs))
    f = torch.cat(feats)
    return f / f.norm(dim=-1, keepdim=True)

train_img_feat = encode_images(train_df["path"].tolist()).to(DEVICE)
gallery_feat = encode_images(gallery_df["path"].tolist()).to(DEVICE)
print("image features:", tuple(train_img_feat.shape), tuple(gallery_feat.shape))

## 2. Train mapper — InfoNCE + pseudo-label smoothing

Text input: `[BOS, pseudo-token, w2, w3, ...]` — pseudo-token ở **vị trí 1** thay từ đầu tiên của caption, embedding của nó do `mapper(image_embedding)` sinh ra. CLIP text encoder bị đóng băng; gradient chỉ chảy về mapper qua embedding được chèn vào.

In [ ]:
import torch.nn.functional as F

def text_forward(img_emb, ids, att):
    """Chèn pseudo-token (từ mapper) vào vị trí 1, trả text features (joint space, normalized)."""
    pseudo = mapper(img_emb)                                   # [B, D]
    emb = clip.text_model.embeddings.token_embedding(ids).clone()  # [B, 77, D]
    emb[:, 1, :] = pseudo
    out = clip.text_model(inputs_embeds=emb, attention_mask=att)
    tfeat = clip.text_projection(out.pooler_output)
    return tfeat / tfeat.norm(dim=-1, keepdim=True)

enc_all = tok(train_df["caption"].tolist(), padding="max_length",
              truncation=True, max_length=77, return_tensors="pt")
opt = torch.optim.AdamW(mapper.parameters(), lr=CFG["LR"])
logit_scale = clip.logit_scale.exp().detach()

n = len(train_df)
for ep in range(CFG["EPOCHS"]):
    perm = torch.randperm(n)
    tot, nb = 0.0, 0
    for i in range(0, n, CFG["BATCH"]):
        idx = perm[i:i + CFG["BATCH"]]
        ie = train_img_feat[idx]
        ids = enc_all["input_ids"][idx].to(DEVICE)
        att = enc_all["attention_mask"][idx].to(DEVICE)
        if random.random() < CFG["SMOOTH_P"]:  # smoothing: caption của ảnh KHÁC trong batch
            ids = ids.roll(1, dims=0)
            att = att.roll(1, dims=0)
        tfeat = text_forward(ie, ids, att)
        logits = logit_scale * ie @ tfeat.t()
        labels = torch.arange(len(idx), device=DEVICE)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    print(f"epoch {ep + 1}/{CFG['EPOCHS']}  loss={tot / nb:.4f}")
print("TRAIN DONE")

## 3. Eval — self-retrieval trên 200 ảnh held-out

Đo Recall@1/5: dùng chính ảnh làm query (ảnh → pseudo-token → câu template), tìm lại đúng ảnh đó trong gallery. So sánh mapper đã train vs baseline nhét **raw** image embedding vào vị trí pseudo-token (2 không gian khác nhau nên tệ — cho thấy vì sao cần mapper).

In [ ]:
@torch.no_grad()
def eval_retrieval(use_mapper=True):
    B = gallery_feat.shape[0]
    enc = tok([CFG["TEMPLATE"]] * B, padding="max_length", truncation=True,
              max_length=77, return_tensors="pt")
    q = gallery_feat
    if use_mapper:
        tfeat = text_forward(q, enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    else:
        emb = clip.text_model.embeddings.token_embedding(enc["input_ids"].to(DEVICE)).clone()
        emb[:, 1, :] = q
        out = clip.text_model(inputs_embeds=emb, attention_mask=enc["attention_mask"].to(DEVICE))
        tfeat = clip.text_projection(out.pooler_output)
        tfeat = tfeat / tfeat.norm(dim=-1, keepdim=True)
    ranks = (tfeat @ gallery_feat.t()).argsort(dim=1, descending=True)
    gt = torch.arange(B, device=DEVICE)
    r1 = (ranks[:, 0] == gt).float().mean().item()
    r5 = (ranks[:, :5] == gt[:, None]).any(1).float().mean().item()
    return r1, r5

import time
t0 = time.time()
R_mapper = eval_retrieval(use_mapper=True)
R_raw = eval_retrieval(use_mapper=False)
print(f"mapper (trained)  : R@1={R_mapper[0]:.3f}  R@5={R_mapper[1]:.3f}")
print(f"raw embed (no map): R@1={R_raw[0]:.3f}  R@5={R_raw[1]:.3f}")
print(f"eval xong trong {time.time() - t0:.1f}s")

## 4. Demo truy xuất + xuất output

Lưu ý thẳng: bản mini chỉ train để pseudo-prompt khớp **ảnh gốc**, nên template đúng nghĩa ZS-CIR ("a sketch of *", sửa thuộc tính) sẽ trôi — bài gốc train cả CC3M (~3M caption) mới tổng quát được. Đây là demo cơ chế, không phải kết quả SOTA.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def query_one(qi, template):
    q = gallery_feat[qi:qi + 1]
    enc = tok([template], padding="max_length", truncation=True, max_length=77, return_tensors="pt")
    tfeat = text_forward(q, enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    sims = (tfeat @ gallery_feat.t())[0]
    top = sims.argsort(descending=True)[:4]
    return top.tolist(), sims[top].tolist()

def demo_grid(qi, template, fname):
    top, ss = query_one(qi, template)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4.2))
    axes[0].imshow(Image.open(gallery_df["path"].iloc[qi]).convert("RGB"))
    axes[0].set_title("QUERY", fontsize=10); axes[0].axis("off")
    for r, (j, s) in enumerate(zip(top, ss), start=1):
        axes[r].imshow(Image.open(gallery_df["path"].iloc[j]).convert("RGB"))
        axes[r].set_title(f"#{r} sim={s:.3f}", fontsize=9); axes[r].axis("off")
    fig.suptitle(template.replace(".", "<pseudo>"), fontsize=11)
    fig.savefig(fname, bbox_inches="tight", dpi=90)
    plt.close(fig)
    print("saved", fname)

random.seed(CFG["SEED"])
for k, qi in enumerate(random.sample(range(len(gallery_df)), 3)):
    demo_grid(qi, CFG["TEMPLATE"], f"demo_q{k}_photo.png")
print("DEMO DONE")

In [ ]:
import json, shutil, pathlib

W = pathlib.Path("/kaggle/working")
metrics = {
    "model": CFG["MODEL"],
    "train_images": int(len(train_df)),
    "gallery_images": int(len(gallery_df)),
    "epochs": CFG["EPOCHS"],
    "recall_at1_mapper": R_mapper[0],
    "recall_at5_mapper": R_mapper[1],
    "recall_at1_raw_baseline": R_raw[0],
    "recall_at5_raw_baseline": R_raw[1],
}
(W / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
torch.save(mapper.state_dict(), W / "mapping.pt")
summary = "\n".join(
    [f"PIC2WORD MINI — {metrics['train_images']} anh train, {metrics['gallery_images']} anh gallery",
     f"R@1 mapper={metrics['recall_at1_mapper']:.3f}  R@5={metrics['recall_at5_mapper']:.3f}",
     f"R@1 raw   ={metrics['recall_at1_raw_baseline']:.3f}  R@5={metrics['recall_at5_raw_baseline']:.3f}"])
(W / "summary.txt").write_text(summary + "\n")
print(summary)
print("OUTPUT READY")